In [ ]:
print('TEST')

In [ ]:
## 이 노트북은 RAG 테스트 용도 입니다.
## 일반적으로 웹에서 사용하는 용도가 아닌, 프로그램을 만들어서 LLM모델을 사용하려고 한다
## LLM Model: Gemini(google) - 무료
## 컨셉: 메뉴데이터를 벡터데이터베이스 구축 한 후 사용자 질의에 응답하는 챗봇

In [ ]:
# LLM 모델 테스트
# 우리가 사용할 LLM 모델은 Gemini
# 그래서 구글 AI STUDIO에서 키를 발급 받아야한다. 이 키를 api key
# 그리고 이 키를 로컬에서 관리한다. .env 폴더를 먼저 만든다 위치 꼭 기억 
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()   #.env에 있는 구글 api 키를 자동으로 로딩

# 모델을 선정하고 파라미터 값을 설정할 수 있다.
llm = ChatGoogleGenerativeAI(model = "gemini-3.5-flash", 
                             temperature= 0.3,
                             max_output_tokens = 1024 # 대답에 대한 토큰 수 설정. 토큰을 판단하는 기준.
                             )
                             # 0.0: 일관성 높은 답변, 0.3 : 0.0보다는 창의성 추가, 일반적인 답변 
                             # 1.0: 기본값 - 창의적인 답변.. 1.5: 정말 창의적인 답변
                             # 창의적인 것을 llm은 단어와 의도를 보고 상황마다 답변이 달라 질 수 있음

# response = llm.invoke('대한민국 소개해 줘')
# print(response.content)

# 모델이 정상 동작임을 확인함.. RAG에서 G부분에서 사용할 모델 테스트 완료 됨.

# 연규할 항목 : 모델별로 성능 차이가 있다.. 개발 주제와 가장 적당한 모델을 찾는 것..
# 주제는 이렇고, 데이터는 이렇고. 그래서 open ai 사의 gpt X, google Gemini 3.5x버전 등을 사용
## 어떤 모델의 토큰량과 정답도를 고려하여 최종 000모델로 선택한다.

In [ ]:
## RAG 처럼 테스트
query = "짜장면은 달고 짜면서 맛있어. 짬뽕은 얼큰하고 맛있어.. 이게 내가 주는 데이터야.. 이 데이터로 광고문구 만들어줘"
response = llm.invoke(query)
print(response.content)

In [ ]:
# 1. LLM에게 전달 할때 우리가게(음식가게)의 메뉴의 정보를 제공해야 합니다
## 그러기 위해서 ..메뉴의 정보를 불러오고 > 벡터데이터로 구축해야한다.
## 우리가게 정보는 menus_recommendation.json에 있음
import json
from langchain_core.documents import Document   # 벡터디비 만들때는 데이터를 Document로 만들어야함
# 벡터디비 만들때는 데이터(우리는 json)를 Document로 만들어야함
# Document 객체는 키의 이름이 page_content와 metadata로 되어있음 (단순 암기 영역)

# 우리의 json 데이터 로드
with open("../storage/menus_recommendation.json", "r", encoding="utf-8") as file:
    data = json.load(file)
# print(data)

# 일부만 확인해보자
#print(data[:1])

# 데이터를 가져왔으니.. 벡터데이터베이스를 만들기 전에 Documents 문서 객체로 변환
# Document생성
docs = []   # document 문서를 저장할 배열
for row in data:
    #content 변수는 문자열이다
    content = f"""
        이름 : {row['name']}
        설명 : {row['description']}
    """
    docmake = Document(
        page_content=content.strip()
    )
    docs.append(docmake)
print(docs[:2]) # 리스트에는 document 타입의 객체가 저장되어 있음 

In [ ]:
# 위에서 생성한 documnet가 저장된 docs리스트로 벡터 데이터베이스를 생성해야 한다
# 이때 docs문장에 있는 한글을 숫자화 해야 한다
# 가장 좋은 성능을 보이는 트랜스포머로 만든 모델을 사용할 것이다.
# 한글을 벡터화 시키는 것을 임베딩모델이라고 하고, 우리가 사용할 임베딩 모델은
# 트랜스포머 방식으로 개발된 모델이다
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},  # GPU 있으면 "cuda"
    encode_kwargs={"normalize_embeddings": True} # 벡터 비교에서 중요한 건 “방향”임
)


In [ ]:
## 벡터데이터베이스를 구축한다
## RAG에서 사용하는 벡터데이터베이스는 정형데이터베이스랑 다르다
## 종류는 pincorn faiss chroma 디비 등이 있다. 나는 chroma db를 사용할것이다
## 그래서 디비가 저장된 디렉토리를 생성함 .. chroma_db


## 벡터 데이터베이스 생성
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=docs,     ## 벡터 데이터 베이스를 구축할 정보
    embedding=embeddings,   ## docs의 문장을 해석하여 데이터베이스를 구축할 임베딩 객체
    persist_directory="../chroma_db/menus_db" ## 데이터베이스 위치
)

In [ ]:
## 벡터 데이터베이스 구축된 정보 확인
print("저장된 문서 개수:" , vectorstore._collection.count())

In [ ]:
## 벡터데이터베이스를 한 번 구축하면 다음에는 구축 안해도 되고 읽어 오기만 하면 된다
## 이미 구축된 데이터베이스에서 데이터를 읽어 오는 법
vectorstore = Chroma(
    persist_directory="../chroma_db/menus_db",
    embedding_function=embeddings   # 주의할 점.. 벡터디비 구축 시 embedding 모델과
                                    # 읽어 올 때의 모델은 동일해야 한다
)

In [ ]:
## 지금까지 작업으로 선정한 데이터(json)으로 벡터데이터베이스를 구축함
## 사용자 질의로 벡터데이터베이스와 유사도가 높은 문서로 검색해보자

## 사용자질문
query = "매콤한 음식 추천"
results = vectorstore.similarity_search(query, k=4)  

for doc in results:
    print(doc.page_content)

In [ ]:
## 이 셀의 목적은 거리 검색의 정확도를 분석해서 품질을 높이기 위함
## 그래서 스코어를 확인하기 위해 similarity_search_with_score메서드를 사용함
## similarity_search 메서드로 검색했는데
## similarity_search_with_score 메서드로 검색해 보자 (거리까지 리턴)
def search_menus(query, k=3):
    result = vectorstore.similarity_search_with_score(query, k=k)
    output = []
    for doc, score in result:
        output.append({
            "내용": doc.page_content,
            "거리": score
        })
    return output

## 테스트
query = "만두를 바삭하게 구운 음식 소개해줘"
results = search_menus(query,4)
for r in results:
    print('거리', r['거리'])
    print('내용',r['내용'])
    print('--------------------')

# 0.0 완전히 동일   0.2: 매우 유사      1.0 : 보통      5.0 전혀 안 비슷

In [ ]:
## 위의 코드는 데이터베이스에서 가져온 값입니다.
## 포인트는 거리 검색을 해서 가져왔다는 것..벡터데이터베이스로 구축이 되어야 함..
## 벡터데이터베이스로 구축하려면 한글을 쪼개서 분석한다. 이 모델이 임베딩모델..(트랜스포머 기반으로 개발된)
## 개발자는 모델을 개발하겠느냐.. 이미 개발된 모델로 제품을 만드는가?

## 컨셉 데이터베이스는 회사의 자료입니다.. 이것을 벡터디비로 구축해서
## if문 등으로 동일한 문장이 아닌.. 유사한 문장을 찾아 내는 방식..
## 검색하는 입장에서 매우 편하겠다..

## 이제는 이 문장을 그대로 사용자에게 보여줄 것인가? 아니면 LLM 모델을 사용해서 
## 회사가 원하는 컨셉으로 문장을 다시 만들것인가?

## 컨셉은 거리 검색한 문장과 사용자 질문으로 LLM에게 문장을 만들어 달라고 할 예정

In [ ]:
## 거리가 짧은 데이터를 검색해 오는 함수 정의
def retrieval_docs(query, k=3):
    results = vectorstore.similarity_search(query, k=k)
    return results

In [ ]:
# 보강 - 검색한 정보를 모델에 입력하기 위해 보강한다.
def format_docs(docs):
    data = ''
    for doc_data in docs:
        data = data + "\n\n" + doc_data.page_content
    print(data)
    return data

In [ ]:
## 질문과 거리 검색과 보강의 결과를 확인해보자
query = "만두를 바삭하게 구운 음식 소개해줘"
format_docs(retrieval_docs(query,3))

In [ ]:
# Generation(생성) - 위에서 만든 보강된 데이터를 이용하여 프롬프트를 만들고 LLM에게 질의
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda # 파이프라인 조립할때 필요한 객체
from langchain_core.output_parsers import StrOutputParser # LLM 응답시 문자열로 하라는 객체
## LLM 응답시 JSON 방식으로 응답이 가능하고 문자열로도 응답이 가능하다
## 이번 예제에서는 간단하게 문자열로 응답하도록 작성함

## 프롬프트 정의
prompt = ChatPromptTemplate.from_template("""
[역할] 음식을 추천하고 설명하는 사장님이야.
[사용자질문] {query}
[참고정보] {context}
[규칙]
- 참고정보만 참고해서 답변을 만들 것. 새로운 음식 추가하지 않는다.
- 음식이름을 먼저 이야기하고 구매 가능하도록 설명한다.
- 설명은 2줄 이하로 한다
- 아주 친절한 모드로 응답한다.
""")

## 파이프 조립을 해보자
## 1. 질의를 통해서 거리 검색 후 보강   2.프롬프트에 query와 context를 대입해서 완성
## 3. LLM에게 질의를 합니다     4. 응답을 받아옵니다.

## 아래처럼 파이프라인을 만들면 전체의 구조가 한눈이 바로 보입니다.
## 문법에 익숙해지자
## prompt, LLM, StrOutputParser는 langChain에서 가져온 객채라서
## 랭체인을 통과하는데 문제가 없다
rag_chain = (
    # 첫번쨰 파이프 라인  
    {
        "query":lambda x : x['usequery'],
        "context": RunnableLambda(lambda x: format_docs(retrieval_docs(x['usequery'])))
        #RunnableLambda langChain을 통과하기 위한 객체
    } 
    |prompt # 두번째 파이프 라인
    |llm # 세번째 파이프 라인
    |StrOutputParser() # 네번째 파이프 라인
)

result = rag_chain.invoke({'usequery':'만두를 바삭하게 구은 음식 소개해 줘'})
print('LLM 생성한 문장입니다.')
print(result)